# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library, guided by the Croissant metadata schema.

### Dataset Source

The dataset is described using a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the FAIR² dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# URL for the Croissant metadata
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Inspect the dataset metadata (metadata is an object, not a dict)
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview

List all available record sets, their `@id`s, with their fields and columns by `@id`.

In [ ]:
# Print all record sets, their @id, and contained fields/columns
record_set_infos = []
print("Available record sets and their fields:")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet name: {getattr(record_set, 'name', '[unnamed]')}")
    print(f"  @id: {record_set.id}")
    # For data extraction, we will need these @ids
    field_ids = []
    print(f"  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - {getattr(field, 'name', '[unnamed]')} (@id: {field.id})")
        field_ids.append(field.id)
        # If field has columns, print column @ids as well
        for col in getattr(field, 'columns', []):
            print(f"      Column: {col.name} (@id: {col.id})")
    print()
    record_set_infos.append({
        'id': record_set.id,
        'name': getattr(record_set, 'name', ''),
        'fields': field_ids
    })

## 3. Data Extraction

Load data for each record set found above, referencing all by their `@id`. All data will be loaded into pandas DataFrames for analysis.

For demonstration, we select the first available record set for display.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs['id'] for rs in record_set_infos]
dataframes = {}

# Load data for each record set by record set @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

if record_set_ids:
    # Work with the first record set as an example
    first_record_set_id = record_set_ids[0]
    print(f"Fields/columns in record set '@id': {first_record_set_id}\n")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets available in the metadata.")

## 4. Exploratory Data Analysis (EDA)

Let's perform basic filtering, normalization, and grouping using a numeric field from the dataset.

If available, we'll use the first numeric column found in the first record set.

In [ ]:
# Identify a numeric field for analysis in the first record set
numeric_field_id = None
group_field_id = None
example_df = dataframes.get(first_record_set_id)
if example_df is not None:
    # Try to automatically pick the first numeric column for demonstration
    for col in example_df.columns:
        if pd.api.types.is_numeric_dtype(example_df[col]):
            numeric_field_id = col
            break
    # As a group field, try to use a likely categorical column
    for col in example_df.columns:
        if col != numeric_field_id and example_df[col].dtype == object and example_df[col].nunique() < 20:
            group_field_id = col
            break

if numeric_field_id is None:
    print("No numeric field found for EDA in this record set.")
else:
    print(f"Using numeric field: {numeric_field_id}")
    threshold = example_df[numeric_field_id].mean() if not np.isnan(example_df[numeric_field_id].mean()) else 0
    filtered_df = example_df[example_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df.head())

    # Normalize column (avoid SettingWithCopyWarning for demonstration)
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() > 0 else 1)
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If there's a suitable group field, show mean grouping
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by '{group_field_id}':")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and, if available, compare distributions across group categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and example_df is not None:
    plt.figure(figsize=(10, 5))
    sns.histplot(example_df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id exists, plot grouped boxplots
    if group_field_id is not None:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=example_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion

This notebook demonstrated how to explore the FAIR² dataset using the `mlcroissant` library by referencing all data entities by their Croissant `@id`. 
- Dataset metadata, record sets, and fields can be systematically explored via Croissant schema.
- Data for each record set can be loaded and processed for further analytics—including filtering, normalization, and visualization—while maintaining clear provenance to the Croissant metadata via each entity's `@id`.

This methodology provides transparent, reproducible steps for working with complex FAIR datasets.